# Regression Analysis — What Drives Option Bid-Ask Spreads?

## What this notebook adds

The statistical tests in `stats.ipynb` answered *whether* differences exist between groups. Regression answers a harder question: **after controlling for everything else that affects spreads, how much did the tariff shock independently contribute?**

For example, NVDA had wider spreads during Phase 2 — but NVDA options also tend to have shorter DTE and higher volatility. Regression isolates the phase effect from those confounders.

We build three models in order of complexity:

| Model | Purpose |
|-------|---------|
| **OLS (main effects)** | Interpretable baseline — how much does each variable shift spreads? |
| **OLS (interactions)** | Tests whether the shock hit different sectors differently, controlling for everything |
| **Random Forest** | Better predictive accuracy — and feature importance shows what matters most |

**Dataset:** all liquid (bid > 0) contracts with valid realised volatility — 49,473 rows across all tickers, phases, and moneyness levels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


PROJECT_ROOT = Path().resolve().parent
COMBINED_CSV = PROJECT_ROOT / "data" / "combined" / "combined_all.csv"

plt.rcParams.update({
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "font.size":         10,
})


# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv(
    COMBINED_CSV,
    parse_dates=["collection_date"],
    dtype={"ticker": str, "side": str, "moneyness_cat": str},
)
df["is_illiquid"] = df["is_illiquid"].astype(bool)

liquid = df[~df["is_illiquid"]].copy()

# Drop rows missing key features
reg_df = liquid.dropna(subset=["relative_spread", "realised_vol", "dte", "moneyness_pct"])
reg_df = reg_df[reg_df["relative_spread"] > 0].copy()

print(f"Regression dataset: {len(reg_df):,} rows")
print(f"Tickers : {sorted(reg_df['ticker'].unique())}")
print(f"Phases  : {sorted(reg_df['phase'].unique())}")

## Feature Engineering

### Target variable: `log(relative_spread)`

Relative spread is right-skewed — most contracts have moderate spreads but a few have very wide ones. Taking the log makes the distribution roughly normal, which is an assumption of OLS. It also changes how we interpret coefficients: a coefficient of 0.5 means the variable is associated with spreads being `e^0.5 ≈ 1.65×` wider, not 0.5 wider.

### Predictor variables

| Variable | What it captures |
|----------|------------------|
| `ticker` dummies | Structural spread differences between sectors (AAPL = reference) |
| `phase_2`, `phase_3` | Effect of the tariff shock and recovery vs baseline (Phase 1 = reference) |
| `realised_vol` | Higher volatility → market makers widen quotes to manage risk |
| `log_dte` | Short-dated options have wider relative spreads; log handles the non-linear decay |
| `abs_moneyness_pct` | Distance from ATM — deep ITM/OTM options are less liquid |
| `is_put` | Whether the option is a put (1) or call (0) |
| `log_oi` | Higher open interest → more trading activity → tighter spreads |
| `log_vol` | Higher daily volume → more active market → tighter spreads |

In [ ]:
# ── Target ────────────────────────────────────────────────────────────────────

reg_df["log_relative_spread"] = np.log(reg_df["relative_spread"])

# ── Predictors ────────────────────────────────────────────────────────────────

# log(dte): clip at 1 to avoid log(0) for same-day expiry contracts
reg_df["log_dte"] = np.log(reg_df["dte"].clip(lower=1))

# Distance from ATM -- absolute value so ITM and OTM are treated symmetrically
reg_df["abs_moneyness_pct"] = reg_df["moneyness_pct"].abs()

# Binary: put = 1, call = 0
reg_df["is_put"] = (reg_df["side"] == "put").astype(int)

# log(1 + x) handles zeros gracefully -- a contract with 0 open interest
# or 0 volume on collection day shouldn't become -inf
reg_df["log_oi"]  = np.log1p(reg_df["openInterest"])
reg_df["log_vol"] = np.log1p(reg_df["volume"])

# Phase dummies -- Phase 1 is the reference (baseline)
reg_df["phase_2"] = (reg_df["phase"] == 2).astype(int)
reg_df["phase_3"] = (reg_df["phase"] == 3).astype(int)

print("Features ready.")
print()
print("Target distribution (log_relative_spread):")
print(reg_df["log_relative_spread"].describe().round(3).to_string())

---
## Model 1 — OLS: Main Effects

This is the baseline model. Each coefficient tells you the expected change in `log(relative_spread)` associated with a one-unit increase in that variable, **holding everything else constant**.

**Reference categories:**
- Ticker: AAPL (so `ticker_NVDA` means "compared to AAPL")
- Phase: Phase 1 / baseline (so `phase_2` means "compared to pre-shock")
- Side: call (so `is_put` means "compared to calls")

**Key question:** Is `phase_2` positive and significant after controlling for all other variables? If yes, the shock widened spreads independently of volatility, DTE, and moneyness.

In [ ]:
formula_1 = (
    "log_relative_spread ~ "
    "C(ticker, Treatment('AAPL')) + "
    "phase_2 + phase_3 + "
    "realised_vol + log_dte + abs_moneyness_pct + "
    "is_put + log_oi + log_vol"
)

ols1 = smf.ols(formula_1, data=reg_df).fit()

# Build a clean results table -- the default statsmodels summary is verbose
results_ols1 = pd.DataFrame({
    "Coefficient":  ols1.params.round(4),
    "Std Error":    ols1.bse.round(4),
    "p-value":      ols1.pvalues.round(4),
    "[95% CI low]": ols1.conf_int()[0].round(4),
    "[95% CI hi]": ols1.conf_int()[1].round(4),
    "Sig":          ols1.pvalues.apply(
        lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    ),
})

print("OLS Model 1 — Main Effects")
print("=" * 75)
print(results_ols1.to_string())
print()
print(f"R²     = {ols1.rsquared:.4f}")
print(f"Adj R² = {ols1.rsquared_adj:.4f}")
print(f"N      = {int(ols1.nobs):,}")

### Reading the coefficients

Because the target is log-transformed, coefficients are interpreted as **multiplicative effects on the spread**. The conversion is:

$$\text{spread multiplier} = e^{\text{coefficient}}$$

So a coefficient of `0.10` means spreads are `e^{0.10} ≈ 1.11×` wider (11% wider). A coefficient of `-0.30` means spreads are `e^{-0.30} ≈ 0.74×` as wide (26% tighter).

**What to look for in the output:**
- `phase_2` coefficient: the shock's independent effect on spreads — positive means the event widened spreads even after controlling for volatility and contract characteristics
- `C(ticker)[T.NVDA]`, `C(ticker)[T.PG]`, etc.: structural spread differences between sectors vs AAPL
- `realised_vol`: does higher market volatility predict wider spreads?
- `log_dte`: do shorter-dated contracts have wider spreads?
- `abs_moneyness_pct`: does moving away from ATM widen spreads?

In [ ]:
# Convert key coefficients to multipliers for easier interpretation
print("Key coefficient interpretation (spread multiplier = e^coef):")
print()

key_vars = [
    ("phase_2",                          "Tariff shock (Phase 2 vs baseline)"),
    ("phase_3",                          "Recovery period (Phase 3 vs baseline)"),
    ("C(ticker, Treatment('AAPL'))[T.NVDA]", "NVDA vs AAPL (structural)"),
    ("C(ticker, Treatment('AAPL'))[T.AMZN]", "AMZN vs AAPL (structural)"),
    ("C(ticker, Treatment('AAPL'))[T.PG]",   "PG vs AAPL (structural)"),
    ("C(ticker, Treatment('AAPL'))[T.CAT]",  "CAT vs AAPL (structural)"),
    ("realised_vol",                     "Realised volatility (+1 unit)"),
    ("log_dte",                          "Log DTE (+1 unit)"),
    ("abs_moneyness_pct",                "Distance from ATM (+1%)"),
    ("is_put",                           "Put vs Call"),
]

print(f"  {'Variable':<45} {'Coef':>7}  {'Multiplier':>10}  {'Sig'}")
print("  " + "-" * 72)
for var, label in key_vars:
    if var in ols1.params:
        coef = ols1.params[var]
        mult = np.exp(coef)
        sig  = results_ols1.loc[var, "Sig"]
        print(f"  {label:<45} {coef:>+7.4f}  {mult:>10.4f}x  {sig}")

---
## Model 2 — OLS: Asymmetric Shock (Interaction Terms)

Model 1 assumes the tariff shock affected all tickers equally (a single `phase_2` coefficient). But the EDA and statistical tests suggested NVDA was hit harder than others.

Model 2 adds **`ticker × phase_2` interaction terms**. This lets each ticker have its own Phase 2 effect. The interaction coefficient for NVDA answers: *"over and above NVDA's structural spread premium, how much extra widening did it experience during the shock?"*

This is the regression equivalent of the pairwise tests, but now controlling for realised volatility, DTE, and moneyness simultaneously.

In [ ]:
formula_2 = (
    "log_relative_spread ~ "
    "C(ticker, Treatment('AAPL')) * phase_2 + "
    "phase_3 + "
    "realised_vol + log_dte + abs_moneyness_pct + "
    "is_put + log_oi + log_vol"
)

ols2 = smf.ols(formula_2, data=reg_df).fit()

# Extract only the interaction terms and phase_2 baseline for readability
interaction_rows = [
    idx for idx in ols2.params.index
    if "phase_2" in idx
]

results_ols2 = pd.DataFrame({
    "Coefficient":  ols2.params[interaction_rows].round(4),
    "p-value":      ols2.pvalues[interaction_rows].round(4),
    "Multiplier":   np.exp(ols2.params[interaction_rows]).round(4),
    "Sig":          ols2.pvalues[interaction_rows].apply(
        lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    ),
})

print("OLS Model 2 — Phase 2 effects and ticker interactions")
print("=" * 65)
print(results_ols2.to_string())
print()
print(f"R²     = {ols2.rsquared:.4f}  (Model 1 R² = {ols1.rsquared:.4f})")
print(f"Adj R² = {ols2.rsquared_adj:.4f}")
print(f"N      = {int(ols2.nobs):,}")
print()
print("Interaction interpretation:")
print("  'phase_2' = shock effect for AAPL (the reference ticker)")
print("  'C(ticker)[T.NVDA]:phase_2' = NVDA's shock effect ON TOP OF the AAPL baseline")
print("  Total shock effect for NVDA = phase_2 coef + C(ticker)[T.NVDA]:phase_2 coef")

In [ ]:
# Compute the total Phase 2 shock effect per ticker (baseline + interaction)
print("Total Phase 2 shock effect per ticker (after controlling for all other variables):")
print()

base_phase2 = ols2.params["phase_2"]
tickers_other = ["NVDA", "AMZN", "PG", "CAT"]

print(f"  {'Ticker':<8} {'Total coef':>12}  {'Multiplier':>12}  {'vs AAPL baseline'}")
print("  " + "-" * 55)

# AAPL: just the base phase_2 coefficient
print(f"  {'AAPL':<8} {base_phase2:>+12.4f}  {np.exp(base_phase2):>12.4f}x  (reference)")

for ticker in tickers_other:
    interaction_key = f"C(ticker, Treatment('AAPL'))[T.{ticker}]:phase_2"
    if interaction_key in ols2.params:
        interaction_coef = ols2.params[interaction_key]
        total_coef = base_phase2 + interaction_coef
        sig = "***" if ols2.pvalues[interaction_key] < 0.001 else (
              "**" if ols2.pvalues[interaction_key] < 0.01 else (
              "*" if ols2.pvalues[interaction_key] < 0.05 else "ns"))
        print(f"  {ticker:<8} {total_coef:>+12.4f}  {np.exp(total_coef):>12.4f}x  "
              f"(interaction {interaction_coef:+.4f}, {sig})")

---
## Model 3 — Random Forest

OLS assumes a linear relationship between each predictor and the log-spread. Random Forest makes no such assumption — it can capture non-linear effects and interactions automatically. The trade-off is interpretability: we can't read off individual coefficients, but we get **feature importance**, which tells us which variables are most useful for predicting spreads.

We use an 80/20 train/test split to get an honest estimate of how well the model generalises to unseen data. R² on the test set is the key number — if it's much lower than on the training set, the model is overfitting.

**Feature importance** is measured by how much each variable reduces prediction error when used as a split in the trees, averaged across all trees. Higher = more important.

In [ ]:
# Build feature matrix manually (RF needs numeric inputs, not formula strings)
ticker_dummies = pd.get_dummies(reg_df["ticker"], prefix="ticker").drop(columns=["ticker_AAPL"])

FEATURE_COLS = [
    "phase_2", "phase_3",
    "realised_vol",
    "log_dte",
    "abs_moneyness_pct",
    "is_put",
    "log_oi",
    "log_vol",
]

X = pd.concat([reg_df[FEATURE_COLS].reset_index(drop=True),
               ticker_dummies.reset_index(drop=True)], axis=1)
y = reg_df["log_relative_spread"].reset_index(drop=True)

all_features = X.columns.tolist()

# 80/20 split -- random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {len(X_train):,} rows")
print(f"Test set     : {len(X_test):,} rows")
print(f"Features     : {len(all_features)}")
print()
print("Fitting Random Forest (100 trees)...")

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,        # limit depth to reduce overfitting
    min_samples_leaf=20, # each leaf must have at least 20 observations
    random_state=42,
    n_jobs=-1,           # use all CPU cores
)
rf.fit(X_train, y_train)

y_pred_train = rf.predict(X_train)
y_pred_test  = rf.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test  = r2_score(y_test,  y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

print(f"Train R² = {r2_train:.4f}  |  Train RMSE = {rmse_train:.4f}")
print(f"Test  R² = {r2_test:.4f}  |  Test  RMSE = {rmse_test:.4f}")

In [ ]:
# Feature importance plot
importances = (
    pd.Series(rf.feature_importances_, index=all_features)
    .sort_values(ascending=True)
)

# Rename for readability on the chart
label_map = {
    "phase_2":           "Phase 2 (shock)",
    "phase_3":           "Phase 3 (recovery)",
    "realised_vol":      "Realised volatility",
    "log_dte":           "Log DTE",
    "abs_moneyness_pct": "Distance from ATM",
    "is_put":            "Is put",
    "log_oi":            "Log open interest",
    "log_vol":           "Log volume",
    "ticker_NVDA":       "Ticker: NVDA",
    "ticker_AMZN":       "Ticker: AMZN",
    "ticker_PG":         "Ticker: PG",
    "ticker_CAT":        "Ticker: CAT",
}
importances.index = [label_map.get(i, i) for i in importances.index]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#E53935" if "Phase 2" in i else
          "#FB8C00" if "Phase 3" in i else
          "#2196F3" if "Ticker" in i else
          "#78909C"
          for i in importances.index]
importances.plot.barh(ax=ax, color=colors, edgecolor="none")
ax.set_title("Random Forest — Feature Importance\n(fraction of variance explained by each variable)",
             fontsize=12, pad=10)
ax.set_xlabel("Importance (mean decrease in impurity)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

from matplotlib.patches import Patch
legend_handles = [
    Patch(color="#E53935", label="Event phase"),
    Patch(color="#2196F3", label="Ticker (structural)"),
    Patch(color="#78909C", label="Contract characteristics"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)
fig.tight_layout()

# save(fig, "fig_rf_importance.png")
plt.show()

---
## Model Comparison

In [ ]:
# OLS in-sample RMSE for comparison
ols1_resid = ols1.resid
ols2_resid = ols2.resid
rmse_ols1 = np.sqrt((ols1_resid ** 2).mean())
rmse_ols2 = np.sqrt((ols2_resid ** 2).mean())

comparison = pd.DataFrame({
    "Model":          ["OLS (main effects)", "OLS (interactions)", "Random Forest"],
    "R²":             [round(ols1.rsquared, 4),
                       round(ols2.rsquared, 4),
                       round(r2_test, 4)],
    "Adj R²":         [round(ols1.rsquared_adj, 4),
                       round(ols2.rsquared_adj, 4),
                       "—"],
    "RMSE":           [round(rmse_ols1, 4),
                       round(rmse_ols2, 4),
                       round(rmse_test, 4)],
    "N":              [f"{int(ols1.nobs):,}",
                       f"{int(ols2.nobs):,}",
                       f"{len(X_test):,} (test)"],
    "Note":           ["In-sample", "In-sample", "Out-of-sample"]
}).set_index("Model")

print(comparison.to_string())
print()
print("Note: OLS metrics are in-sample; RF R² and RMSE are on the held-out 20% test set.")
print("      A higher RF R² does not imply OLS is wrong — they answer different questions.")
print("      OLS gives interpretable coefficients; RF gives predictive accuracy.")

---
## Key Findings from the Regression

In [ ]:
print("KEY FINDINGS")
print("=" * 70)
print()

# Phase 2 effect from Model 1
phase2_coef = ols1.params["phase_2"]
phase2_mult = np.exp(phase2_coef)
phase2_sig  = ols1.pvalues["phase_2"]
print("1. Tariff shock independently widened spreads (Model 1)")
print(f"   phase_2 coef = {phase2_coef:+.4f}  →  spreads {phase2_mult:.3f}x wider during Phase 2")
print(f"   p = {phase2_sig:.4f}  {'***' if phase2_sig < 0.001 else '**' if phase2_sig < 0.01 else '*' if phase2_sig < 0.05 else 'ns'}")
print(f"   This holds after controlling for volatility, DTE, moneyness, and sector.")
print()

# Phase 3 effect
phase3_coef = ols1.params["phase_3"]
phase3_mult = np.exp(phase3_coef)
print("2. Recovery: Phase 3 spreads vs baseline (Model 1)")
print(f"   phase_3 coef = {phase3_coef:+.4f}  →  spreads {phase3_mult:.3f}x vs baseline in Phase 3")
print(f"   {'Spreads overshot baseline (tighter than pre-shock) even after controls.' if phase3_coef < 0 else 'Spreads still elevated in Phase 3 after controls.'}")
print()

# NVDA interaction
nvda_int_key = "C(ticker, Treatment('AAPL'))[T.NVDA]:phase_2"
if nvda_int_key in ols2.params:
    nvda_int = ols2.params[nvda_int_key]
    nvda_total = ols2.params["phase_2"] + nvda_int
    nvda_sig = ols2.pvalues[nvda_int_key]
    print("3. NVDA shock premium over AAPL (Model 2 interaction)")
    print(f"   NVDA interaction coef = {nvda_int:+.4f}")
    print(f"   Total Phase 2 effect for NVDA = {nvda_total:+.4f}  ({np.exp(nvda_total):.3f}x)")
    print(f"   p = {nvda_sig:.4f}  — {'significant: NVDA absorbed a larger shock than AAPL after controls' if nvda_sig < 0.05 else 'not significant after controlling for other variables'}")
print()

# Most important features from RF
top3 = importances.sort_values(ascending=False).head(3)
print("4. Most important predictors of spread (Random Forest)")
for feat, imp in top3.items():
    print(f"   {feat:<30} {imp:.4f} ({imp*100:.1f}% of variance)")
print()

print(f"5. Model fit")
print(f"   OLS R² = {ols1.rsquared:.4f} — linear controls explain {ols1.rsquared*100:.1f}% of log-spread variance")
print(f"   RF test R² = {r2_test:.4f} — non-linear model explains {r2_test*100:.1f}% out of sample")